# Download, Decompress, Parse, Insert into Database, and Delete

In [1]:
import psycopg2
import os
import shutil
import sys
import datetime

# redirect the output to a log file
logname = (
    "logs/" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_wikitest" + ".log"
)
log = open(logname, "a")
sys.stdout = log

errname = (
    "logs/" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_wikitest" + ".err"
)
err = open(errname, "a")
sys.stderr = err
import requests
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import py7zr

# Database connection details
db_host = "localhost"
db_port = "5432"
db_name = "wikitest"
db_user = "richard"
db_password = "rich"

import mwxml
import time
import tqdm

In [2]:
import os

finished = os.listdir("finished_test")
import pandas as pd

dump_names_df = pd.read_csv("wiki240201.csv", header=None)
dump_names = dump_names_df[0].to_list()
urls = [
    "https://dumps.wikimedia.org/enwiki/20240201/" + i
    for i in dump_names
    if i not in finished
]
len(urls)

843

In [3]:
def download_file(url, filename):
    response = requests.get(url, stream=True)
    with open(filename, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file.write(chunk)
    print(f"Downloaded: {filename}")


def decompress_file(filename):
    with py7zr.SevenZipFile(filename, mode="r") as archive:
        archive.extractall()
    print(f"Decompressed: {filename}")


def insert_db(data_list):
    status = False
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host=db_host,
            port=db_port,
            dbname=db_name,
            user=db_user,
            password=db_password,
        )

        # Create a cursor object
        cur = conn.cursor()

        # Create the tables (same as before)
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY,
                text VARCHAR NULL,
                deleted BOOLEAN DEFAULT FALSE
            );
        """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY,
                title VARCHAR NULL,
                namespace INTEGER NULL,
                restrictions JSONB NULL,
                deleted BOOLEAN DEFAULT FALSE
            );
        """
        )
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS revisions (
                id INTEGER PRIMARY KEY,
                timestamp TIMESTAMP NULL,
                user_id INTEGER NULL REFERENCES users(id),
                page_id INTEGER NULL REFERENCES pages(id),
                minor BOOLEAN NULL,
                comment TEXT NULL,
                text TEXT NULL,
                bytes INTEGER NULL,
                sha1 VARCHAR NULL,
                model VARCHAR NULL,
                format VARCHAR NULL,
                deleted_text BOOLEAN DEFAULT FALSE,
                deleted_comment BOOLEAN DEFAULT FALSE,
                deleted_user BOOLEAN DEFAULT FALSE
            );
        """
        )
        # Insert user data
        user_data = [
            (
                d.get("user", {}).get("id"),
                d.get("user", {}).get("text"),
                d.get("deleted", {}).get("user"),
            )
            for d in data_list
            if d.get("user", {}).get("id") is not None
        ]
        cur.executemany(
            """
            INSERT INTO users (id, text, deleted)
            VALUES (%s, %s, %s)
            ON CONFLICT (id) DO NOTHING;
        """,
            user_data,
        )

        # Insert page data
        page_data = [
            (
                d.get("page", {}).get("id"),
                d.get("page", {}).get("title"),
                d.get("page", {}).get("namespace"),
                d.get("page", {}).get("restrictions"),
                False,
            )
            for d in data_list
            if d.get("page", {}).get("id") is not None
        ]
        cur.executemany(
            """
            INSERT INTO pages (id, title, namespace, restrictions, deleted)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING;
        """,
            page_data,
        )

        # Insert revision data
        revision_data = [
            (
                d.get("id"),
                d.get("timestamp"),
                d.get("user", {}).get("id"),
                d.get("page", {}).get("id"),
                d.get("minor"),
                d.get("comment"),
                d.get("text"),
                d.get("bytes"),
                d.get("sha1"),
                d.get("model"),
                d.get("format"),
                d.get("deleted", {}).get("text"),
                d.get("deleted", {}).get("comment"),
                d.get("deleted", {}).get("user"),
            )
            for d in data_list
            if d.get("id") is not None
        ]
        cur.executemany(
            """
            INSERT INTO revisions (id, timestamp, user_id, page_id, minor, comment, text, bytes, sha1, model, format, deleted_text, deleted_comment, deleted_user)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING;
        """,
            revision_data,
        )

        # Commit the changes
        conn.commit()
        status = True
        print("Data inserted successfully.")

    except psycopg2.Error as e:
        # Handle any errors that occur during the database operations
        print(f"An error occurred: {e}")
        conn.rollback()

    finally:
        # Close the cursor and the database connection
        if cur:
            cur.close()
        if conn:
            conn.close()
        return status


def parse_insert(dump_name):
    print(
        f"Start Dump {dump_name}. Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}"
    )
    dump = mwxml.Dump.from_file(open(dump_name, "rb"))
    data_list = []
    total_number = 0
    part_number = 0

    for page in dump:
        if page.namespace in [1, 3, 5, 7, 9, 11, 13, 101, 119, 711, 829, 2301, 2303]:
            for revision in page:
                data_dict = revision.to_json()
                data_list.append(data_dict)
                part_number += 1
        if part_number >= 10:
            print(
                f"\t\t\tInserting {part_number} records. Total: {total_number + part_number}"
            )
            insert_db(data_list)
            print(
                f"\t\t\tInserted {part_number} records. Total: {total_number + part_number}"
            )
            time.sleep(1)
            print(
                f"\t\t\tCurrent Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}",
                flush=True,
            )
            total_number += part_number
            if total_number >= 20:
                break
            data_list = []
            part_number = 0

    if data_list:
        insert_db(data_list)
        print(f"Inserted {part_number} records. Total: {total_number + part_number}")
        time.sleep(5)
        print(f"Current Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}")
        total_number += part_number
        data_list = []
        total_number += part_number

    print(
        f"Completed Dump {dump_name}. Total: {total_number} records. Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}"
    )


def delete_file(dump_name):
    os.remove(dump_name)
    print(f"Deleted: {dump_name}")


def move_file_to_finished_directory(dump_name):
    source_file = dump_name + ".7z"
    destination_directory = "finished_test"
    shutil.move(source_file, destination_directory)
    print(f"Moved {source_file} to {destination_directory} directory.")


def ddpid(url):
    filename = url.split("/")[-1]
    download_file(url, filename)
    decompress_file(filename)
    dump_file_name = filename.rstrip(".7z")
    parse_insert(dump_file_name)
    delete_file(dump_file_name)
    move_file_to_finished_directory(dump_file_name)

In [4]:
if __name__ == "__main__":
    dump_names_df = pd.read_csv("wiki240201.csv", header=None)
    dump_names = dump_names_df[0].to_list()
    urls = ["https://dumps.wikimedia.org/enwiki/20240201/" + i for i in dump_names]

with tqdm.tqdm(total=len(urls)) as pbar:
    with ThreadPoolExecutor() as executor:
        for _ in executor.map(ddpid, urls[:1]):
            pbar.update(1)